# Proyecto Implementacion fase 1
- Javier Prado 21486
- Bryan España 21550

## Dataset a utilizar
El dataset a utilizar es sacado de:
https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
Este es el conjunto de datos de evaluación de detección de intrusiones (CIC-IDS2017). 

## Construcción del dataset (CIC-IDS2017)

El dataset CIC-IDS2017 fue construido por el Canadian Institute for Cybersecurity (CIC) con el objetivo de crear un conjunto de datos realista y actualizado para la evaluación de sistemas de detección de intrusiones (IDS). A diferencia de datasets antiguos, este fue diseñado para simular el comportamiento real de una red empresarial bajo condiciones normales y bajo ataque.

## Cómo fue construido

El dataset se generó en un entorno controlado de red durante un período de aproximadamente 5 días (lunes a viernes), donde se simuló el tráfico de una organización real. Durante este proceso:

Se configuró una infraestructura con múltiples máquinas (atacantes y víctimas) que representaban una red empresarial real.
Se generó tráfico normal (benigno) utilizando perfiles de comportamiento de usuarios reales, simulando actividades como navegación web, envío de correos, uso de FTP y SSH.
A partir del segundo día, se ejecutaron distintos tipos de ataques controlados en la red, incluyendo:
Ataques de fuerza bruta (FTP y SSH)
Ataques DoS y DDoS
Escaneo de puertos
Ataques web (SQL Injection, XSS)
Botnets e infiltraciones
Heartbleed

Todo el tráfico de red fue capturado en formato PCAP (packet capture), lo que representa los paquetes reales transmitidos en la red.

## Procesamiento de los datos

Posteriormente, los datos capturados fueron procesados utilizando una herramienta llamada CICFlowMeter, que transforma el tráfico de red en flujos (flows).

Un flow representa una comunicación entre dos dispositivos (IP origen y destino), y de cada flujo se extraen múltiples características como:

Número de paquetes
Duración del flujo
Tamaño de paquetes
Flags TCP
Estadísticas (media, desviación estándar, etc.)

Esto permitió generar alrededor de 80 características por flujo.

Características del conjunto de datos

![Descripción](imagen.png)

## Labels

Cada flujo fue etiquetado como:

Benigno (normal)
Ataque (con tipo específico)

En total, el dataset contiene múltiples clases de ataque además de la clase normal, lo que lo hace ideal para problemas de clasificación supervisada en Machine Learning.

# Analisis exploratorio

In [ ]:
# Para descargar el dataset se debe descargar con el siguiente codigo
# se mantendra comentada esta seccion para evitar descargar el dataset cada vez que se ejecute el codigo

# al ejecutarlo se creara una carpeta en la carpeta del repo, dentro de esta carpeta se descargara el dataset
from pathlib import Path
import kagglehub
import shutil

# 📁 Ruta base (donde está el notebook)
BASE_DIR = Path().resolve()

# 📁 Carpeta dataset dentro del proyecto
DATASET_DIR = BASE_DIR / "dataset"
DATASET_HANDLE = "chethuhn/network-intrusion-dataset"

# Crear carpeta si no existe
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Evitar descargar si ya existe contenido
if any(DATASET_DIR.iterdir()):
    print("⚡ Dataset ya existe en:", DATASET_DIR)
else:
    print("📥 Descargando dataset...")

    # Descargar dataset con el identificador completo de Kaggle
    download_path = Path(kagglehub.dataset_download(DATASET_HANDLE))

    # Copiar archivos desde la caché de Kaggle al proyecto
    for source in download_path.iterdir():
        destination = DATASET_DIR / source.name
        if source.is_dir():
            shutil.copytree(source, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(source, destination)

    print("✅ Dataset descargado en:", DATASET_DIR)

c:\Users\javil\OneDrive\Documentos\U\Data Science\Intrusion_Detection_System\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 230M/230M [00:09<00:00, 25.7MB/s] 

Extracting files...


Path to dataset files: C:\Users\javil\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1


## Analisis exploratorio detallado

Este bloque construye un EDA orientado a describir el dataset **CIC-IDS2017** desde cuatro angulos:

- estructura general del conjunto de datos
- distribucion de clases y archivos
- calidad de datos y variables redundantes
- caracteristicas clave mas informativas para distinguir trafico benigno y ataques


In [ ]:
from cicids_eda import run_eda

artifacts = run_eda(DATASET_DIR)


In [ ]:
# Tabla de caracteristicas clave para el reporte
feature_report = (
    artifacts["key_features"][["feature", "category", "description", "security_value", "mutual_info", "attack_vs_benign_shift"]]
    .rename(columns={
        "feature": "Caracteristica",
        "category": "Categoria",
        "description": "Descripcion",
        "security_value": "Valor para IDS",
        "mutual_info": "Mutual information",
        "attack_vs_benign_shift": "Cambio ataque vs benigno",
    })
)

feature_report


In [ ]:
from IPython.display import Markdown, display

overview = artifacts["overview"].set_index("metric")["value"]
label_distribution = artifacts["label_distribution"].copy()
quality = artifacts["quality"].copy()
duplicates = artifacts["duplicates"].copy()

top_attacks = label_distribution[label_distribution["label"] != "BENIGN"].head(3)
top_attacks_text = ", ".join(
    f"{row.label} ({row.pct:.2f}%)"
    for row in top_attacks.itertuples(index=False)
)

rare_classes = label_distribution[label_distribution["pct"] < 0.01]["label"].tolist()
rare_classes_text = ", ".join(rare_classes) if rare_classes else "ninguna"

quality_notes = []
for row in quality.itertuples(index=False):
    parts = []
    if row.missing_values:
        parts.append(f"{int(row.missing_values)} nulos")
    if row.infinite_values:
        parts.append(f"{int(row.infinite_values)} infinitos")
    quality_notes.append(f"{row.feature}: {'; '.join(parts)}")

duplicate_total = int(duplicates["duplicate_rows"].sum())
quality_notes_text = ", ".join(quality_notes) if quality_notes else "sin problemas de calidad visibles"

display(Markdown(f"""
## Resumen interpretativo del EDA

**Contexto del dataset.** Segun la documentacion oficial del CIC, el trafico fue capturado entre el **3 y el 7 de julio de 2017**. El **lunes** contiene solo trafico benigno y entre **martes y viernes** se introducen ataques de fuerza bruta, DoS/DDoS, web attacks, botnet, infiltration, heartbleed y port scanning.

**Hallazgos principales sobre tu copia del dataset descargada desde Kaggle.**
- El conjunto analizado contiene **{int(overview["Registros totales"]):,} registros**, **{int(overview["Caracteristicas numericas y de red"])} variables** y **{int(overview["Clases de trafico"])} clases**.
- Existe **desbalance de clases**: `BENIGN` representa **{label_distribution.iloc[0]["pct"]:.2f}%** del total. Las familias de ataque mas frecuentes son **{top_attacks_text}**.
- Las clases extremadamente raras son **{rare_classes_text}**, por lo que conviene tratarlas con cuidado al graficar o entrenar modelos.
- Las variables con mas poder descriptivo se concentran en **duracion del flujo**, **tasas por segundo**, **tiempos entre llegadas**, **tamano de paquetes** y **longitud de cabeceras**.
- En calidad de datos se observan **{quality_notes_text}** y **{duplicate_total:,} filas duplicadas** dentro de archivos.
- Antes de modelar, conviene **limpiar infinitos**, **eliminar columnas constantes o redundantes** y usar una estrategia para el **desbalance de clases**.

**Fuentes base para contextualizar el dataset.**
- UNB CIC: https://www.unb.ca/cic/datasets/ids-2017.html
- Kaggle: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset/data
"""))
